# Garbage Detector — Full Training Pipeline
**Run this entire notebook top to bottom on Kaggle.**

What it does:
1. Installs dependencies
2. Downloads a real garbage/litter dataset via Open Images v7 (reliable, no account needed)
3. Converts to YOLO format
4. Fine-tunes YOLOv8
5. Shows accuracy metrics + sample predictions
6. Zips `best.pt` so you can download it

**Before running:** Top-right → `Session options` → `Accelerator: GPU T4 x1` → Save

## Step 1 — Install dependencies

In [ ]:
!pip install ultralytics fiftyone --quiet

import ultralytics
ultralytics.checks()  # confirms GPU is visible

## Step 2 — Download garbage dataset from Open Images v7

Open Images is Google's large-scale dataset. We download only images that contain
`Trash` annotations — no account or API key needed.

We pull **800 train + 200 val** images. You can raise `MAX_TRAIN_SAMPLES` for better
accuracy at the cost of longer download time.

In [ ]:
import fiftyone as fo
import fiftyone.zoo as foz

MAX_TRAIN_SAMPLES = 800
MAX_VAL_SAMPLES   = 200

print("Downloading train split (~800 images with Trash labels)...")
train_dataset = foz.load_zoo_dataset(
    "open-images-v7",
    split="train",
    label_types=["detections"],
    classes=["Trash"],
    max_samples=MAX_TRAIN_SAMPLES,
    dataset_name="oi_trash_train",
)
print(f"Train images loaded: {len(train_dataset)}")

print("\nDownloading validation split (~200 images)...")
val_dataset = foz.load_zoo_dataset(
    "open-images-v7",
    split="validation",
    label_types=["detections"],
    classes=["Trash"],
    max_samples=MAX_VAL_SAMPLES,
    dataset_name="oi_trash_val",
)
print(f"Val images loaded: {len(val_dataset)}")

## Step 3 — Convert to YOLO format

Open Images uses normalised `[x_centre, y_centre, width, height]` internally,
but fiftyone stores them as `[x_top_left, y_top_left, width, height]` relative
to image size. We convert and write the YOLO `.txt` label files.

In [ ]:
import shutil
from pathlib import Path

DATASET_DIR = Path("garbage_dataset")
CLASS_NAMES = ["trash"]   # single class — everything is garbage

def export_split(fo_dataset, split_name):
    img_dir   = DATASET_DIR / "images" / split_name
    label_dir = DATASET_DIR / "labels" / split_name
    img_dir.mkdir(parents=True, exist_ok=True)
    label_dir.mkdir(parents=True, exist_ok=True)

    written = 0
    for sample in fo_dataset:
        src = Path(sample.filepath)
        if not src.exists():
            continue

        # Copy image
        dst_img = img_dir / src.name
        shutil.copy2(src, dst_img)

        # Write label file
        lines = []
        detections = sample.ground_truth
        if detections is None:
            continue
        for det in detections.detections:
            # fiftyone bbox: [x_tl, y_tl, w, h] all normalised 0-1
            x, y, w, h = det.bounding_box
            cx = x + w / 2
            cy = y + h / 2
            # Clamp to [0, 1] to guard against annotation noise
            cx = max(0.0, min(1.0, cx))
            cy = max(0.0, min(1.0, cy))
            w  = max(0.001, min(1.0, w))
            h  = max(0.001, min(1.0, h))
            lines.append(f"0 {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")

        if not lines:
            continue

        (label_dir / (src.stem + ".txt")).write_text("\n".join(lines))
        written += 1

    print(f"  {split_name}: {written} images written")

print("Exporting to YOLO format...")
export_split(train_dataset, "train")
export_split(val_dataset,   "val")

# Sanity check
n_train = len(list((DATASET_DIR / "images" / "train").glob("*.*")))
n_val   = len(list((DATASET_DIR / "images" / "val").glob("*.*")))
print(f"\nFinal count — train: {n_train}  val: {n_val}")
assert n_train > 0, "ERROR: No training images found! Something went wrong in Step 2."
assert n_val   > 0, "ERROR: No validation images found!"
print("All good — ready to train.")

## Step 4 — Write `data.yaml`

In [ ]:
from pathlib import Path

DATASET_DIR = Path("garbage_dataset")

yaml_text = f"""path: {DATASET_DIR.resolve()}
train: images/train
val:   images/val

nc: 1
names: ['trash']
"""

(DATASET_DIR / "data.yaml").write_text(yaml_text)
print(yaml_text)

## Step 5 — Fine-tune YOLOv8

On a Kaggle T4 GPU this takes **~20-40 minutes**.

- `MODEL_SIZE = "n"` → nano, fastest (~20 min)
- `MODEL_SIZE = "s"` → small, better accuracy (~40 min)

In [ ]:
from ultralytics import YOLO
from pathlib import Path

MODEL_SIZE  = "n"
EPOCHS      = 50
BATCH       = 16
IMGSZ       = 640
RUN_NAME    = "garbage_finetune"
DATASET_DIR = Path("garbage_dataset")

model = YOLO(f"yolov8{MODEL_SIZE}.pt")

model.train(
    data      = str(DATASET_DIR / "data.yaml"),
    epochs    = EPOCHS,
    imgsz     = IMGSZ,
    batch     = BATCH,
    name      = RUN_NAME,
    device    = 0,

    # Augmentation tuned for outdoor garbage in varied lighting
    hsv_h     = 0.02,
    hsv_s     = 0.6,
    hsv_v     = 0.4,
    fliplr    = 0.5,
    flipud    = 0.1,
    mosaic    = 1.0,
    scale     = 0.5,
    translate = 0.1,
    degrees   = 10.0,

    patience    = 20,
    save_period = 10,
    plots       = True,
    val         = True,
)

BEST_WEIGHTS = Path(f"runs/detect/{RUN_NAME}/weights/best.pt")
print(f"\nTraining done! Best weights: {BEST_WEIGHTS}")

## Step 6 — Evaluate accuracy

In [ ]:
from ultralytics import YOLO
from pathlib import Path

DATASET_DIR  = Path("garbage_dataset")
BEST_WEIGHTS = Path("runs/detect/garbage_finetune/weights/best.pt")

model   = YOLO(str(BEST_WEIGHTS))
metrics = model.val(data=str(DATASET_DIR / "data.yaml"), imgsz=640)

print("\n===== Validation Results =====")
print(f"mAP50:     {metrics.box.map50:.3f}   (1.0 = perfect)")
print(f"mAP50-95:  {metrics.box.map:.3f}")
print(f"Precision: {metrics.box.mp:.3f}   (how often detections are correct)")
print(f"Recall:    {metrics.box.mr:.3f}   (how often garbage is found)")
print()
print("What these mean:")
print("  mAP50 > 0.5 = decent  |  > 0.7 = good  |  > 0.85 = great")
print("  Recall matters more than precision for a garbage detector")
print("  (missing garbage is worse than a false alarm)")

## Step 7 — Visualise predictions on validation images

In [ ]:
import random
from pathlib import Path
from ultralytics import YOLO
import matplotlib.pyplot as plt
import cv2

DATASET_DIR  = Path("garbage_dataset")
BEST_WEIGHTS = Path("runs/detect/garbage_finetune/weights/best.pt")

model      = YOLO(str(BEST_WEIGHTS))
val_images = list((DATASET_DIR / "images" / "val").glob("*.*"))
samples    = random.sample(val_images, min(6, len(val_images)))

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for ax, img_path in zip(axes, samples):
    results     = model(str(img_path), conf=0.3, verbose=False)[0]
    plotted     = results.plot()
    plotted_rgb = cv2.cvtColor(plotted, cv2.COLOR_BGR2RGB)
    ax.imshow(plotted_rgb)
    ax.set_title(img_path.name, fontsize=8)
    ax.axis("off")

plt.suptitle("Sample Predictions on Validation Set", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("sample_predictions.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved to sample_predictions.png")

## Step 8 — Download your trained weights

After the cell runs, find `garbage_model.zip` in the Kaggle file panel (right side) → click Download.

Then on your Mac:
```bash
unzip garbage_model.zip
python garbage_detector.py --source 0 --weights best.pt
```

In [ ]:
import shutil
from pathlib import Path

BEST_WEIGHTS = Path("runs/detect/garbage_finetune/weights/best.pt")

if BEST_WEIGHTS.exists():
    shutil.copy2(BEST_WEIGHTS, "best.pt")
    shutil.make_archive("garbage_model", "zip", ".", "best.pt")
    size_mb = Path("garbage_model.zip").stat().st_size / 1e6
    print(f"garbage_model.zip created ({size_mb:.1f} MB)")
    print("Find it in the Kaggle file browser on the right → click Download")
else:
    print("best.pt not found — did training complete? Check runs/detect/ directory.")